# CDA5: Fourrier Analysis 

Fourrier analysis is a method to discover what "periods" are most present in data, i.e. in a timeseries (they can also be applied spatially but we will consider, wlog, series of data in the time dimension here only for simplicity). While we will show an application in meteorology, Fourier transforms have found use in almost every field on the planet!  

Here are just a couple of examples I took from betterexplained.com:

- **Earthquakes** If earthquake vibrations can be separated into "ingredients" (vibrations of different speeds & amplitudes), buildings can be designed to avoid interacting with the strongest ones.

- **Filters** If sound waves can be separated into ingredients (bass and treble frequencies), we can boost the parts we care about, and hide the ones we don't. The crackle of random noise can be removed. Maybe similar "sound recipes" can be compared (music recognition services compare recipes, not the raw audio clips).

- **Compression** If computer data can be represented with oscillating patterns, perhaps the least-important ones can be ignored. This "lossy compression" can drastically shrink file sizes (and why JPEG and MP3 files are much smaller than raw .bmp or .wav files).  That being said, most compression methods in use these days use a related wavelet based approach which will be introduced later. 

So Let's start this story with a wave, a simple cosine wave will do...


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math

In [ ]:
def add_wave(ax, x, freq=0.02, amplitude=1, phi=0, samples_per_wavelength=20, label='Cosine Wave'):
    # y = A * sin(2pi * f * x + phi)
    y = amplitude * np.cos(2 * np.pi * freq * x + phi)
    
    # Calculate skip step (pss) based on user's samples_per_wavelength
    # (Points in one wavelength) / (Desired samples in one wavelength)
    wavelength = 1 / freq
    points_per_unit = len(x) / (x[-1] - x[0])
    pss = max(1, int((points_per_unit * wavelength) / samples_per_wavelength))
    
    line, = ax.plot(x, y, label=label)
    ax.plot(x[::pss], y[::pss], 'ro', markersize=6, alpha=0.6,label="sample pts")
    return ax

# Define global X-axis
x_fixed = np.linspace(0, 500, 2000)

# Create the figure and axis
fig, ax = plt.subplots(figsize=(10, 4))

# Plot the first wave
freq=0.02
amplitude=1
add_wave(ax, x_fixed, freq=freq, amplitude=amplitude, label='Wave 1')

ax.legend()
ax.grid(True, linestyle=':', alpha=0.5)

### Discussion
<details>
<summary>1) If we have 10 waves and 20 samples per wave, how many red points are required to prepresent the series?</summary>

Trivial really, if we have 20 red dots per wave, then 200 points are required to represent this curve

</details>

<details>
<summary>
2) If we instead represent the data as a cosine how many numbers are required to represent the data?    
</summary>
We can represent this snippet exactly as part of an infinite cosine wave. And to describe that wave we just need to know the 
    
    * wavelength
    * magnitude 

</details>

## Magnitude and Frequency

So we can represent the cosine wave exactly in a continuous form: $$y=A cos(2 \pi f t) $$ where $f$ is the frequency of the cosine in cycles per unit time, $t$ is the time and $A$ is the amplitude of the wave.  This means that instead of 200 datapoints, we could represent this wave with two real numbers, $A$ and $f$. 

## Phase 

But we are forgetting a detail, because we also have another degree of freedom.  We could shift the curve to the right or left, by a discrete amount of time $t_0$

$$y=A cos(2 \pi f (t + t_0)) $$

By convention we can write this as 

$$y=A cos(2 \pi f (t + t_0)) = A cos (2 \pi t + \phi)$$

where $\phi=2\pi f t_0$ is known as the **phase** in radians.  We can show this by shifting the curve by a 6th of a cycle ($\phi=2\pi/6$)


In [ ]:

plt.close()
fig,ax = plt.subplots(figsize=(10, 4))

add_wave(ax, x_fixed, label='Wave 1')
add_wave(ax, x_fixed, phi=np.pi/3, label='Wave 2 (Shifted)')

# Update the legend to show the new entry
ax.legend()


## The neatness of the Complex Space :-)

So... if someone were to tell you the frequency $f$, phase $\phi$ and magnitude $A$, of the cosine wave $y=M cos(ft+\phi)$ you could perfectly reconstruct the wave.  Alternatively they could provide you with the set of data of x-y points to reconstruct a discrete approximation of the wave. 

But in general we can instead represent this information as a complex number, using the follow trigonometric relation $cos(f+g)=cos(f)cos(g)-sin(f)sin(g)$. This equivalence us to write our cosine as a sum of a cosine and sine:

$$y(t) = A\cos(2\pi f t + \phi) = \underbrace{[A\cos(\phi)]}_{\text{Real Weight}} \cos(2\pi ft) - \underbrace{[A\sin(\phi)]}_{\text{Imaginary Weight}} \sin(2\pi ft)$$

This is very elegant, as now for a given wave frequency $f$, we can define the wave uniquely as a complex number $z = a + b j$.

> **Note:** We use $j$ instead of the more common $i$ for complex numbers as it is standard in engineering contexts where the FFT analysis was originally developed and also because it is the mandatory notation used in python and thus avoids notation confusion!

Here, the real part $a$ is the cosine weight:

$$a=Acos(\phi)$$

and likewise the imaginary part is the sine weight

$$b=Asin(\phi)$$

we can calculate the amplitude of the wave:

$$A=\sqrt{a^2 + b^2}$$

And finally we can calculate the phase using the *4-quadrant inverse tangent* function atan2, as follows

$$-\operatorname{atan2}(b, a)$$

> **Note:** We use `atan2(imag, real)` because it correctly identifies which of the 4 quadrants the phase is in by looking at the signs of $a$ and $b$ individually.

### From Trigonometry to the Complex Plane: Euler's Relation

To understand why the FFT gives us complex numbers ($a + b j$), we use **Euler's Relation**. This formula connects exponential functions to trigonometry:

$$e^{j\theta} = \cos(\theta) + j\sin(\theta)$$

When we analyze a signal $x(t) = A \cos(2\pi f t + \phi)$, the FFT looks for a complex coefficient ($z$) that represents this wave at frequency $f$. We can write our wave in "Complex Phasor" form as:

$$A e^{j\phi} = \underbrace{A \cos(\phi)}_{\text{Real Part (a)}} + j \underbrace{A \sin(\phi)}_{\text{Imaginary Part (b)}}$$

#### Recall: the FFT Output
If the FFT result for a specific frequency is the complex number $z = a + b j$:

1. **The Real Part ($a$):** Tells us the strength of the **Cosine** component ($A \cos \phi$).
2. **The Imaginary Part ($b$):** Tells us the strength of the **Sine** component ($-A \sin \phi$).
3. **The Magnitude ($|z|$):** Use the Pythagorean theorem to find the total Amplitude ($A$):
   $$A = \sqrt{a^2 + b^2}$$
4. **The Phase ($\phi$):** Use the 4-quadrant inverse tangent to find the starting shift:
   $$\phi = -\operatorname{atan2}(b, a)$$


### Multiple cosines + noise...

Okay, we have a single (co)sinewave, but what about if we make it more challenging and make a function out of multiple cosines with white noise on top?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# 1. Parameters
n_modes=10 # we will build up the signal with this many modes from most important first
n_waves=6  #  Our fake signal has n_waves added together plus white noise
noise_mag=0.5

n_points = 1000
x = np.linspace(0, 1, n_points)
np.random.seed(42) # For consistent results, seed it to the answer of life and the universe!

# 2. Create 6 random underlying frequencies
# bounds are hardwired for each category at the moment...
freqs = np.random.uniform(5, 50, n_waves)
amps = np.random.uniform(0.5, 2.0, n_waves)
phases = np.random.uniform(0, 2*np.pi, n_waves)

# Build the "Clean" signal
clean_signal = np.zeros(n_points)
for f, a, p in zip(freqs, amps, phases):
    clean_signal += a * np.cos(2 * np.pi * f * x + p)

# Add significant white noise
noise = np.random.normal(0, noise_mag, n_points)
noisy_signal = clean_signal + noise

# 3. Analyze the signal using FFT to find the modes
fft_values = np.fft.rfft(noisy_signal)
frequencies = np.fft.rfftfreq(n_points, d=x[1]-x[0])

# Get indices of the top 10 strongest magnitudes (modes)
magnitudes = np.abs(fft_values) # SQRT(a**2+ b**2)

# We skip the 0th index (DC offset) and pick the top 10≈
indices = np.argsort(magnitudes[1:])[::-1][:n_modes] + 1 

# Pre-calculate variance of the noisy signal for the EV formula
total_variance = np.var(noisy_signal)

# 4. Set up the Animation Plot
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(x, noisy_signal, color='gray', alpha=0.7, label='Noisy Data')
line, = ax.plot([], [], color='blue', lw=2, label='Fourier Reconstruction')
text = ax.text(0.02, 0.95, '', transform=ax.transAxes, fontweight='bold')

ax.set_xlim(0, 1)
ax.set_ylim(noisy_signal.min()-1, noisy_signal.max()+1)
ax.legend(loc='upper right')

def init():
    line.set_data([], [])
    return line,

def update(frame):
    # Sum up the first 'frame + 1' modes
    current_modes = indices[:frame]
    
    # Create a blank FFT array and only fill in our selected modes
    filtered_fft = np.zeros_like(fft_values)
    for idx in current_modes:
        filtered_fft[idx] = fft_values[idx]
    
    # Inverse FFT to get back the reconstructed time-domain signal
    reconstructed = np.fft.irfft(filtered_fft, n=n_points)

    # Calculate Explained Variance
    if frame == 0:
        ev = 0.0
    else:
        # Residual variance is the variance of the difference
        residual_variance = np.var(noisy_signal - reconstructed)
        ev = (1 - (residual_variance / total_variance)) * 100

    line.set_data(x, reconstructed)
    text.set_text(f"Modes: {frame:2d} | Explained Var: {ev:5.1f}%")

    return line, text

# Create Animation (frames 0 to 9 for 10 modes)
ani = FuncAnimation(fig, update, frames=range(n_modes+1), init_func=init, blit=True, interval=800)

# Display in Notebook
plt.close() # Prevents double plot
HTML(ani.to_jshtml())



## And Sharp Gradients?

So you might be asking yourself: *"That was easy because the data was made up of smooth waves, but what about data with sharp, vertical transitions?"* Interestingly, Fourier theory tells us that functions don't even need to be smooth (infinitely differentiable) to be represented by sinusoids. For example, we can reconstruct an **ideal square wave** with an amplitude of 1 by summing an infinite series of **cosines**:

$$f(t) = \frac{4}{\pi} \sum_{k=1}^{n} \frac{(-1)^{k-1}}{2k - 1} \cos(2\pi(2k - 1)ft)$$

Where:
* **$f$** is the frequency in **Hertz** (cycles per second).
* **$2\pi$** converts our frequency into angular velocity.
* The **$2k-1$** term ensures we only use **odd harmonics** (1st, 3rd, 5th, etc.).
* The **$(-1)^{k-1}$** term alternates the sign of each harmonic to "flatten" the top of the wave.

Let's try it out with a square wave of **one Hertz**!


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def square_wave(t, freq):
    """Generates a square wave with a specific frequency in Hz."""
    # Using Cosine-based sign to align with the Fourier series below
    return np.sign(np.cos(2 * np.pi * freq * t))

def sum_of_cosines(t, freq, num_terms):
    """Approximates a square wave using the Cosine Fourier Series."""
    approximation = np.zeros_like(t, dtype=float)
    for k in range(1, num_terms + 1):
        n = 2 * k - 1  # Only odd harmonics
        # The (-1)^(k-1) flips the sign for every other term to flatten the top
        term = (4 / (np.pi * n)) * ((-1)**(k-1)) * np.cos(n * 2 * np.pi * freq * t)
        approximation += term
    return approximation

# --- Parameters ---
frequency_hz = 1.0        # 1 cycle per second
duration_seconds = 3.0    # Show 3 seconds of data
sample_rate = 1000        # Points per second

time = np.linspace(0, duration_seconds, int(sample_rate * duration_seconds))
square_signal = square_wave(time, frequency_hz)

# --- Plotting ---
num_terms = 5
approximation = sum_of_cosines(time, frequency_hz, num_terms)

fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(time, square_signal, label='Ideal Square Wave', color='gray', linestyle='--', alpha=0.5)
ax.plot(time, approximation, label=f'Fourier Series ({num_terms} Cosine terms)', color='blue')

ax.set_title(f'Square Wave Synthesis ({frequency_hz} Hz)')
ax.set_xlabel('Time (seconds)')
ax.set_ylabel('Amplitude')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Exercise

With start with one term by default, but try increasing this and rerunning the following code...


> <div style="padding: 15px; border: 1px solid #1565C0; border-left: 6px solid #1565C0; background-color: #E3F2FD; color: #0D47A1; border-radius: 4px;">
> <strong> The Gibbs Phenomenon</strong><br>
> Notice the "ringing" or overshooting at the sharp corners of the square wave? This is known as the <strong>Gibbs Phenomenon</strong>. Because we are using continuous, smooth waves (cosines) to approximate a discontinuous jump, the sum will always overshoot the target by about 9%, no matter how many terms you add! It only disappears when you have an <em>infinite</em> number of terms.
> </div>


# Fourier Series

So we shown how a smooth sum of cosines with noise could be recontructed with cosine, no big surprise perhaps...  Then we showed how a square wave could be recontructed and approximated, a bit more surprising. 

But what about other more generalized series and forms?  Well it turns out that *any function* in time (or space) can be represented by a sum of cosines!  When Fourier first introduced this idea it was apparently rejected by many of the leading mathematicians of the idea for quite some time!  But it is true, any function can be represented as a summation of  a series of sines and cosines of different frequencies, called a Fourier Series. 

As we saw above, there are two common forms of the Fourier Series, "Trigonometric" and "Exponential." These are discussed below, followed by a demonstration that the two forms are equivalent.  So for a periodic signal, where $\omega_0$ is the frequency of the periodic signal: 

#### Trigonometric Form (Sum of Sines and Cosines)

$$ x(t) = a_0 + \sum_{n=1}^{\infty} \left[ a_n \cos(n \omega_0 t) + b_n \sin(n \omega_0 t) \right] $$


So essentially Fourier suggested an incredible thing, that essentially we can represent any signal alternatively in a "frequency" (or wavelength) space, as a series of curves of a given wave length.

So now the task is to *"transform"* a continuous function, or perhaps a series of discrete data points, into their equivalent representation as a sum of sines and cosines, that is, their representation in *frequency space*.  This is known as a *Fourier transform*. 

We just need to find out how to calculate the amplitude (and phase) of each sinewave, essentially the series of $a_n$ and $b_n$. 

This is useful because we may find that particular wavelengths/frequencies have much higher amplitude $A$, implying that this frequency is particularly pronounced in the signal, something that is not always immediately obvious when looking at the original signal in space or time $x(t)$.

#### Complex Exponential Form

Equivalently, we can write this in a Complex Exponential Form (Sum of Complex Numbers)
$$x(t) = \sum_{n=-\infty}^{\infty} c_n e^{j n \omega_0 t} $$

The task of deriving the amplitude and phase of each sinewave, essentially the series of $a_n$ and $b_n$, is equivalent to finding the complex coefficients $c_n$

The relationship between the complex coefficient $c_n$ and the real coefficients $a_n$ and $b_n$ comes directly from **Euler's Formula**. This connects our standard trigonometric waves to complex exponentials.

Euler's formula ($e^{j\theta} = \cos\theta + j\sin\theta$) allows us to express cosine and sine entirely in terms of complex exponentials:

$$\cos(\theta) = \frac{e^{j\theta} + e^{-j\theta}}{2}$$

$$\sin(\theta) = \frac{e^{j\theta} - e^{-j\theta}}{2j}$$

Because $\frac{1}{j} = -j$, we can conveniently rewrite the sine formula as:*

$$\sin(\theta) = -j \left( \frac{e^{j\theta} - e^{-j\theta}}{2} \right)$$

Now, replace the $\cos$ and $\sin$ in our original equation with their exponential forms (letting $\theta = n\omega_0 t$):

$$f_n(t) = a_n \left( \frac{e^{jn\omega_0 t} + e^{-jn\omega_0 t}}{2} \right) - j b_n \left( \frac{e^{jn\omega_0 t} - e^{-jn\omega_0 t}}{2} \right)$$

#### Group  by Exponent
Expand the brackets and group together all the terms multiplied by the positive exponent ($e^{jn\omega_0 t}$) and the negative exponent ($e^{-jn\omega_0 t}$):

$$f_n(t) = \left( \frac{a_n}{2} - j\frac{b_n}{2} \right) e^{jn\omega_0 t} + \left( \frac{a_n}{2} + j\frac{b_n}{2} \right) e^{-jn\omega_0 t}$$

Factor out the $\frac{1}{2}$:

$$f_n(t) = \frac{1}{2}(a_n - jb_n) e^{jn\omega_0 t} + \frac{1}{2}(a_n + jb_n) e^{-jn\omega_0 t}$$

#### Equate to the Complex Fourier Series
The Complex Exponential form of the Fourier series is defined as a sum over both positive and negative frequencies:

$$x(t) = \dots + c_{-n} e^{-jn\omega_0 t} + c_n e^{jn\omega_0 t} + \dots$$

By simply comparing our rearranged trigonometric equation to the complex exponential definition, we can match the coefficients to the exponentials:

For the **positive** frequencies ($e^{jn\omega_0 t}$):

$$c_n = \frac{1}{2}(a_n - j b_n)$$

For the **negative** frequencies ($e^{-jn\omega_0 t}$):

$$c_{-n} = \frac{1}{2}(a_n + j b_n)$$

   
### What is a Negative Frequency?

In signal processing, a **negative frequency** is not a "physical" impossibility; it is a mathematical necessity that arises the moment we stop looking at waves as simple "up-and-down" vibrations and start looking at them as **rotations in the complex plane.**

#### 1. The Clock Metaphor: Direction of Rotation
Recall **Euler’s Formula**: $e^{j\theta} = \cos \theta + j \sin \theta$. This represents a point moving around a circle. Frequency ($f$) tells us how many times it circles per second.

* **Positive Frequency ($+f$):** The point rotates **Counter-Clockwise**.
* **Negative Frequency ($-f$):** The point rotates **Clockwise**.

If you only look at the **Real axis** (the horizontal "shadow" of the point), you cannot tell the difference. Both clockwise and counter-clockwise rotations look like a simple back-and-forth oscillation.

Let's illustrate this with a python code that plots a point in the complex plane, and its real projection. You will see how the point in the complex plane rotates in an anticlockwise direction first (positive frequencies) for 3 cycles, and then clockwise (negative frequencies).  The real projection looks the same!


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

def create_signal_engine(t, signals, labels, colors, suptitle_func, wave_configs, show_proj_in_legend=False):
    fig, (ax_l, ax_r) = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)
    
    ax_l.set_xlim(-1.3, 1.3); ax_l.set_ylim(-1.3, 1.3)
    ax_l.set_aspect('equal'); ax_l.grid(True, alpha=0.3)
    ax_l.add_artist(plt.Circle((0, 0), 1, color='gray', fill=False, linestyle='--'))
    
    ax_r.set_xlim(t[0], t[-1]); ax_r.set_ylim(-1.3, 1.3)
    ax_r.grid(True, alpha=0.3)
    
    # 1. Complex Plane Dots
    dots = [ax_l.plot([], [], 'o', color=c, markersize=8, label=l)[0] for c, l in zip(colors, labels)]
    
    # 2. Projection Dot Logic
    # If show_proj_in_legend is False, we set label to "_nolegend_" so it doesn't appear
    proj_label = "Real Projection" if show_proj_in_legend else "_nolegend_"
    proj_dot, = ax_l.plot([], [], 'bo', markersize=6, label=proj_label, alpha=0.6)
    
    # 3. Wave Lines and Marker
    wave_lines = [ax_r.plot([], [], color=cfg['color'], lw=2)[0] for cfg in wave_configs]
    time_marker, = ax_r.plot([], [], 'ro', markersize=8)
    
    ax_l.legend(loc='upper right', fontsize='x-small')
    st = fig.suptitle("", fontsize=14, fontweight='bold')

    def update(frame):
        curr_t = t[frame]
        for i, dot in enumerate(dots):
            dot.set_data([signals[i][frame].real], [signals[i][frame].imag])
        
        proj_dot.set_data([signals[0][frame].real], [0])
        
        for i, cfg in enumerate(wave_configs):
            t_start, t_end = cfg['range']
            mask = (t <= curr_t) & (t >= t_start) & (t <= t_end)
            if np.any(mask):
                wave_lines[i].set_data(t[mask], signals[-1].real[mask])
        
        time_marker.set_data([curr_t], [signals[-1].real[frame]])
        title_text, title_color = suptitle_func(curr_t)
        st.set_text(title_text); st.set_color(title_color)
        
        return dots + [proj_dot] + wave_lines + [time_marker]

    ani = FuncAnimation(fig, update, frames=len(t), interval=60, blit=True)
    plt.close()
    return ani

In [ ]:
fs = 30
ncycle=4
t_total = np.linspace(0, ncycle, fs * ncycle)
# Signal switches from +1Hz to -1Hz at 3 seconds
full_sig = np.where(t_total <= ncycle/2, 
                    np.exp(1j * 2 * np.pi * 1 * t_total), 
                    np.exp(-1j * 2 * np.pi * 1 * (t_total - ncycle/2)))

def title_logic_seq(curr_t):
    if curr_t <= ncycle/2: return "PHASE 1: Positive Frequency (+1 Hz) | CCW", "green"
    return "PHASE 2: Negative Frequency (-1 Hz) | CW", "red"

# Define two separate line segments for the time series
configs_seq = [
    {'range': (0, ncycle/2), 'color': 'green'},
    {'range': (ncycle/2, ncycle), 'color': 'red'}
]

ani1 = create_signal_engine(t_total, [full_sig], ["Complex Path"], ["orange"], 
                            title_logic_seq, configs_seq, show_proj_in_legend=True)
HTML(ani1.to_jshtml())


Mathematically, a pure cosine wave is actually the average of two counter-rotating complex vectors:
$$\cos(\theta) = \frac{e^{j\theta} + e^{-j\theta}}{2}$$

Every "Real" signal is actually composed of two "Ghost" signals spinning in opposite directions. The $+f$ and $-f$ components are mirror images that work together to "cancel out" the imaginary $j$ parts.



In [ ]:
ncycle=3
t_sim = np.linspace(0, ncycle, fs * ncycle)

# positive and negative complex numbers and their average)
s_pos = np.exp(1j * 2 * np.pi * 1 * t_sim)
s_neg = np.exp(-1j * 2 * np.pi * 1 * t_sim)
s_avg = (s_pos + s_neg) / 2

def title_logic_sim(curr_t):
    return r"$\cos(\omega t)$ as the Average of Pos and Neg Frequencies", "black"

# Only one segment needed here
configs_sim = [{'range': (0, ncycle), 'color': 'blue'}]

# (Keep same t_sim, s_pos, s_neg, s_avg definitions from previous message)
ani2 = create_signal_engine(t_sim, 
                            [s_pos, s_neg, s_avg], 
                            ["Pos Freq", "Neg Freq", "Average (Real)"], 
                            ["green", "red", "blue"], 
                            title_logic_sim, configs_sim, show_proj_in_legend=False)
HTML(ani2.to_jshtml())


Usually we will want to apply this to a measured signal that consists of a series of discrete datapoints in space or time. We assume in time in the following wlog for brevity. Thus we introduce the discrete Fourier transform. 

## Discrete Fourier transform (DFT)

The DFT of a series of $N$ data points (where $N$ is typically assumed to be even for simplicity, though DFTs can be calculated for odd $n$) produces $N$ complex numbers.  As we saw early, each complex number corresponds to a specific frequency and describes the weights of the sine and cosine components from which we can derive the "power" (amplitude) of the specific wavelength/frequency in the original signal.   

Let $x[n]$ represent a discrete-time signal, where $n$ is an integer index representing the time sample, and $n = 0, 1, ..., N-1$, with $N$ being the total number of samples. 

To understand what the DFT is actually doing, we can think of it as a **projection**. We are measuring how much of our signal $x[n]$ "matches" a specific reference wave (a basis function).

For a given frequency index $k$, (or frequency $k/N$) our reference wave is a complex sinusoid:
$$\phi_k[n] = e^{j \frac{2\pi k n}{N}}$$
  

The standard way to measure the similarity between two complex sequences (vectors) $\mathbf{a}$ and $\mathbf{b}$ is the **complex inner product** (or dot product). The definition of this product requires the **complex conjugate** ($^*$) of the second vector:

$$\langle \mathbf{a}, \mathbf{b} \rangle = \sum_{n=0}^{N-1} a[n] \cdot b^*[n]$$

Thus, to find the frequency coefficient $X[k]$, we take the dot product of our signal $x[n]$ with our basis function $\phi_k[n]$:

$$X[k] = \sum_{n=0}^{N-1} x[n] \cdot (\phi_k[n])^*$$

The complex conjugate of an exponential $e^{j\theta}$ is simply $e^{-j\theta}$. When we apply this to our basis function, the imaginary part is negated:

$$(\phi_k[n])^* = \left( e^{j \frac{2\pi k n}{N}} \right)^* = e^{-j \frac{2\pi k n}{N}}$$

Substituting this conjugate back into our projection sum, we arrive at the standard definition of the Discrete Fourier Transform:

$$X[k] = \sum_{n=0}^{N-1} x[n] e^{-j \frac{2\pi k n}{N}}$$

where $k = 0, 1, ..., N-1$ represents the frequency index. 

### Convolution:
> Essentially what we are doing is a **convolution** of the signal with the complex wave form, for each wavenumber. You can think of this as essentially a dot product (or a correlation!) between the signal you wish to analyse and a series of complex "test" waves.

Thus the DFT decomposes the signal $x[n]$ into a **sum** of *complex sinusoids*, each with a specific frequency and amplitude. The output $X[k]$ is a complex-valued sequence, where:

- the magnitude $|X[k]|$ is related to the amplitude ("power") of the frequency component $k/N$, 
- and the phase $\arg(X[k])$ represents the phase shift of that component.

### Units of Power/Amplitude

The algorithm produces values that are scaled by the number of samples ($N$). To get the actual physical amplitude (in meters), you have to undo that scaling, otherwise higher  frequencies would falsely have more "weight". 

As the FFT "accumulates" the signal across all $N$ points. Dividing by $N$ gives you the average contribution per sample. Thus for plotting, we want to turn the amplitude, or power, back into units of the original dataset and do this by scaling by a factor of $1/N$. Note that this is by convention, and the $1/N$ factor is placed in the inverse transform (see below).  This is somewhat arbitrary and instead could have been placed in the forward transform and would then be known as teh **Modified DFT** or **Statistical DFT**. 

#### Mirror in your FFT
When you run `fft(signal)`, the output array contains $N$ points. 
1.  **Index $0$:** The DC offset (The Mean).
2.  **Indices $1$ to $N/2$:** The **Positive Frequencies** (rotating anticlockwise).
3.  **Indices $N/2+1$ to $N-1$:** The **Negative Frequencies** (rotating clockwise).

Because in our usual applications our timeseries (e.g. temperatures) are Real-valued, the negative frequencies contain the exact same information as the positive ones. We could take the sum of the postive and the complex conjugate of the negative frequencies, but in general it is faster to simpler discard the negative frequencies and scale the positive frequencies by a factor of 2.   

This is why we usually write `[:N//2]` in our code; we are just hiding the redundant "clockwise" half of the frequencies.

> **Note:** When we plot the *single-sided* amplitude we mustn't forget multiply by 2, so the overall scale factor is thus $\frac{2}{N}$.


#### Notation summary 

* **$a_n$ and $b_n$**: These are the real-valued weights for the cosine (real part) and sine (imaginary part) components of a wave. Together, they can be represented as a single complex number ($z = a + bj$).
  
* **$c_n$**: These are the complex coefficients used in the continuous Complex Exponential Form of the Fourier Series ($x(t)=\sum_{n=-\infty}^{\infty}c_{n}e^{jn\omega_{0}t}$). They are the direct, single-variable complex equivalent to the pairs of $a_n$ and $b_n$. The  relationship for positive frequencies is given by the equation:

  $$c_n = \frac{1}{2}(a_n - j b_n)$$

* **$X[k]$**: This is the notation used for the output of the Discrete Fourier Transform (DFT), which is applied to a finite series of discrete data points rather than a continuous function. $X[k]$ is a complex-valued sequence where each value corresponds to a specific frequency index $k$. In short, $X[k]$ is simply the discrete, computational equivalent of the continuous complex coefficients $c_n$.

### Frequency resolution 

There are two fundamental limits to the frequency range that we sample:

1. Lowest frequency, this is by the length of the dataset.  Essentially the frequency corresponds to **one wavelength** over the length of the dataset
2. Instead, the highest frequency that is representable will be related to the sampling frequency $f_s$ of the signal (the number of samples per unit time). The maximum frequency that can be represented without **aliasing** is the Nyquist frequency, which is half the sampling frequency:

$$f_{Nyquist} = \frac{f_s}{2}$$

### Aliasing 

Given a sampling rate $f_s$, any variability in the underlying signal that is occuring on frequencies higher than $f_{Nyquist}$, will be *projected* falsely onto longer wavelength in a process know as **aliasing**. An example is given in the next code block.

In [ ]:
#--------------
# Aliasing Demo
#--------------

# Note: 1 / 1.3 samples per wavelength = 1 sample every 1.3 waves
alias_samples = 1 / 1.3  

fig, ax = plt.subplots(figsize=(10, 4))

# Plot the original wave (blue) with red dots showing the aliasing
add_wave(ax, x_fixed, freq=0.02, amplitude=1, 
         samples_per_wavelength=alias_samples, 
         label='Original 0.02Hz Wave')

# Connect the dots to reveal the "Fake" (Aliased) low-frequency wave
# We slice the same way the function does internally to get the points
wavelength = 1 / 0.02
points_per_unit = len(x_fixed) / (x_fixed[-1] - x_fixed[0])
pss = int((points_per_unit * wavelength) / alias_samples)

ax.plot(x_fixed[::pss], np.cos(2 * np.pi * 0.02 * x_fixed[::pss]), 
        'r--', alpha=0.5, label='Aliased Signal (Fake)')

ax.set_title(f"Aliasing: Sampling at 1 point every 1.3 wavelengths")
ax.legend(loc='upper right', fontsize='small')
plt.show()



So to summarize, the amplitude spectrum of the signal is obtained by computing the magnitude $|X[k]|$ of the DFT coefficients and the power spectrum is given by $|X[k]|^2$.

The frequency corresponding to the $k$-th DFT coefficient is:

$$f_k = k \Delta f = \frac{k f_s}{N}$$

The inverse DFT (IDFT) reconstructs the original signal $x[n]$ from its DFT coefficients $X[k]$:

$$x[n] = \frac{1}{N} \sum_{k=0}^{N-1} X[k] e^{j 2\pi kn/N} $$

For a **real-valued input signal** $x[n]$, the DFT output $X[k]$ exhibits conjugate symmetry, i.e., $X[k] = X^*[N-k]$, where $*$ denotes complex conjugation. This means that the negative frequency components ($k > N/2$) are redundant, and we typically focus on the positive frequency components ($k \le N/2$).

Let's code this up:

In [ ]:
from numba import njit

@njit

def dft_slow(signal, fs=1.0, onlyhalf=True):
    """
    Calculates the Discrete Fourier Transform using a Matrix Multiplication.
    
    Args:
        signal (np.ndarray): The input time-series data.
        fs (float): The sampling frequency (samples per second).
        onlyhalf (logical): True only return half the frequencies (default)
                            False, returns all frequencies to show symmetry when input real
        
    Returns:
        freqs (np.ndarray): Positive frequency axis.
        amplitudes (np.ndarray): Normalized real-world amplitudes.
        phases (np.ndarray): Phase angle in radians.
    """
    N = len(signal)
    
    # 1. The DFT Matrix Math
    n = np.arange(N)
    k = n.reshape((N, 1)) # Transform to column vector for broadcasting
    
    # Create the Basis Matrix: W[k, n] = e^(-j * 2pi * k * n / N)
    W = np.exp(-2j * np.pi * k * n / N)
    
    # Compute the dot product (The Summation)
    signal_complex = signal.astype(np.complex128)
    dft_complex = np.dot(W, signal_complex)
    
    # 2. Extract the 'Useful Stuff' (Positive frequencies only)
    half_n = N // 2
    dft_half = dft_complex[:half_n]
    
    # Frequency Axis: (fs / k * N)
    freqs = np.arange(half_n) * (fs / N)
    
    # Amplitude Normalization: 
    # Multiply by 2/N for harmonics, but only 1/N for the constant (0Hz) bin
    amplitudes = (2.0 / N) * np.abs(dft_half)
    amplitudes[0] = amplitudes[0] / 2.0 
    
    # Phase
    phases = np.angle(dft_half)
    
    return freqs, amplitudes, phases

## Fourier Series on a more challenging example function

Let's imagine a function that is periodic and NOT made up of sinewaves... :-) 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 1. Perfectly Aligned Parameters
points_per_cycle = 256  # power of 2 is nice.
num_cycles = 5
N=num_cycles*points_per_cycle
time = np.linspace(0, num_cycles * 4, N, endpoint=False)

def generate_smooth_periodic_signal_alt(t):
    """Generates a smooth periodic signal using different functions (no sin/cos, no noise)."""
    signal = np.zeros_like(t, dtype=float)
    for i, time_val in enumerate(t):
        phase = time_val % 4
        if 0 <= phase < 1:
            signal[i] = phase**2  # Quadratic rise
        elif 1 <= phase < 2:
            signal[i] = 2 - (phase - 1)**2  # Inverted quadratic peak
        elif 2 <= phase < 3:
            signal[i] = -(phase - 2)**2 # quadratic fall
        else:
            signal[i] = (phase-3)**2 - 1 # quadratic dip
    return signal

# Generate the signal
smooth_periodic_signal = generate_smooth_periodic_signal_alt(time)

# Plotting
plt.figure(figsize=(10, 6))
plt.plot(time, smooth_periodic_signal)
plt.title("Smooth Periodic Signal (Quadratic Segments)")
plt.xlabel("Time (time units, e.g. seconds)")
plt.ylabel("Amplitude")
plt.grid(True)
plt.show()

let's define a small helper function for the scipy FFT:

In [ ]:

def fft_scipy(time,signal):
    ### SCIPY function 
    N   = len(time)
    dt  = time[1] - time[0]
    cn  = fft(signal)
    frq = fftfreq(N, dt)[:N//2]
    amps    = (2.0/N) * np.abs(cn[:N//2])
    amps[0] = amps[0]/2.0         # freq 0 has no mirror!
    phases  = np.angle(cn[:N//2]) # arg for phase
    return frq,cn,amps,phases


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.fft import fft, fftfreq 

dt=time[1]-time[0]

# scipy function
frq_scipy,cn_scipy,amps_scipy,phase_scipy=fft_scipy(time,smooth_periodic_signal)

# our function from Scratch using convolution 
frq, amps, phi = dft_slow(smooth_periodic_signal, fs=1.0/dt)

# Overplot Comparison
plt.figure(figsize=(12, 5))

# Plot Scipy FFT in Blue (The 'Base' line)
plt.stem(frq_scipy, amps_scipy, linefmt='C0-', markerfmt='C0o', 
         basefmt=" ", label='Scipy FFT, Fast O(NlogN)')

# Plot Scratch DFT in Red with a tiny shift (e.g., +0.02 Hz) to the right
# This allows you to see the red bars standing next to the blue bars
shift = 0.02 
plt.stem(frq + shift, amps, linefmt='red', markerfmt='ro', 
         basefmt=" ", label='DFT with slow convolution O(N^2)')

plt.title("Comparison: Scipy FFT (fast! NlogN) vs. Manual Matrix DFT (slow N**2)")
plt.xlabel("Frequency (Cycles per Time Unit)")
plt.ylabel("Amplitude")
plt.xlim(0, 5) 
plt.legend()
plt.grid(alpha=0.3)
plt.show()

# Quick Sanity Check Print
diff = np.abs(amps_scipy - amps).max()
print(f"Maximum difference between methods: {diff:.2e}")

Let's interpret this. The first X axis point refers to frequency ZERO - this is the "DC" offset, i.e. the mean of the data. If the data had had the mean removed then the freq=0 would have an amplitude of zero.

Then there are 3 points with amplitude of zero, followed by the 4th point with the maximum peak 



## Wavenumber

In many data processing contexts, "wavenumber" refers to Cycles per Record. If your dataset is $L$ units long, then:$$\text{Wavenumber } (k) = f \times L$$Where $k=4$ literally means 4 complete sine waves fit into your specific window of data.

Well they look identical so that is good.  

#### Discuss 
> The power sits in discrete frequencies, why is that do you think?  Do you think natural timeseries will behave like this?

### Efficiency and the Fast Fourier transform (FFT)

By the way, we’ve built a mathematically perfect Matrix DFT, but as you'll see, Linear Algebra is expensive. While our "from scratch" version is elegant and readable :-), the Scipy FFT uses the Cooley-Tukey algorithm, which turns a "check every point against every frequency" ($O(N^2)$) problem into a "divide and conquer" ($O(N \log N)$) problem.  For large datasets this is critical...

The FFT is an efficient algorithm to compute the DFT and IDFT. It leverages the symmetry and periodicity properties of the complex exponential term to reduce the computational complexity from $O(N^2)$ for the direct DFT calculation to $O(N \log N)$.

Recall also the FFT in scipy is compiled in C, but we have also compiled the "slow" version using Numba...


In [ ]:
import timeit
import matplotlib.pyplot as plt

# 1. Setup the Benchmark
# We run the FFT 1000 times because it's so fast
# We run the Scratch DFT only 50 times because it's heavy
N_loops_fft = 1000
N_loops_scratch = 50

# Time the Scipy FFT
fft_timer = timeit.Timer(lambda: fft(smooth_periodic_signal))
fft_times = fft_timer.repeat(repeat=5, number=N_loops_fft)
avg_fft_time = min(fft_times) / N_loops_fft  # Best time per operation

# Time the Scratch DFT
scratch_timer = timeit.Timer(lambda: dft_slow(smooth_periodic_signal, fs=1/dt))
scratch_times = scratch_timer.repeat(repeat=5, number=N_loops_scratch)
avg_scratch_time = min(scratch_times) / N_loops_scratch

print(f"The Manual DFT is approximately {avg_scratch_time / avg_fft_time:.0f}x slower than the FFT.")

so let's use the FFT on our bespoke function

In [ ]:
def create_fft_animation(time_axis, signal, title_suffix=""):
    xf, yf, amplitudes, phases = fft_scipy(time_axis, signal)
    
    fig, (ax_spec, ax_time) = plt.subplots(2, 1, figsize=(10, 7), 
                                           gridspec_kw={'height_ratios': [1, 2]})
    
    # --- SETUP STATIC ELEMENTS ONCE (Outside animate) ---
    ax_spec.set_xlim(0.01, 5)
    ax_spec.set_ylim(0, np.max(amplitudes) * 1.2)
    
    # Vectorized functions for the secondary axis
    def freq2period(x): 
        # Avoid division by zero using np.where
        return np.divide(1, x, out=np.full_like(x, np.inf), where=x!=0)

    def period2freq(x): 
        return np.divide(1, x, out=np.full_like(x, np.inf), where=x!=0)

    # Create the secondary axis ONLY ONCE
    secax = ax_spec.secondary_xaxis('top', functions=(freq2period, period2freq))
    secax.set_xlabel('Period [s]', fontsize=9, color='tab:blue')
    secax.set_ticks([0.2, 0.5, 1, 2, 5])

    plt.close() 

    def animate(i):
        # Use cla() but we must re-apply limits/labels because cla() wipes them
        ax_spec.cla()
        ax_time.cla()
        
        # 1. Upper Panel
        ax_spec.stem(xf[:i+1], amplitudes[:i+1], basefmt="C0-", linefmt='C0-', markerfmt='C0o')
        ax_spec.set_xlim(0.01, 5)
        ax_spec.set_ylim(0, np.max(amplitudes) * 1.2)
        ax_spec.set_xlabel("Frequency (Hz)")
        ax_spec.set_title(f"Power Spectrum: Adding Mode {i} {title_suffix}", fontsize=10)
        ax_spec.grid(alpha=0.2)

        # 2. Lower Panel
        current_recon = np.zeros_like(time_axis)
        for k in range(i + 1):
            current_recon += amplitudes[k] * np.cos(2 * np.pi * xf[k] * time_axis + phases[k])
        
        ax_time.plot(time_axis, signal, color='gray', alpha=0.3, label='Actual Signal')
        ax_time.plot(time_axis, current_recon, color='tab:red', lw=2, label=f'Sum of {i+1} Components')
        
        ax_time.set_ylim(np.min(signal)-0.5, np.max(signal)+0.5) 
        ax_time.legend(loc='upper left', fontsize='small')
        ax_time.grid(alpha=0.3)

    num_frames = min(50, len(xf))
    return FuncAnimation(fig, animate, frames=num_frames, interval=150, blit=False)

In [ ]:
# 1. Clean Signal Animation
ani_clean = create_fft_animation(time, smooth_periodic_signal, title_suffix="(Clean)")
HTML(ani_clean.to_jshtml())

we can see how the sinewaves are already starting to reproduce the curve with 25 modes, and that certain modes are much more "important" that others for this particular function. 

There is also an offset, what do you think is causing that?  We can make things even worse  by adding a trend 


In [ ]:
# 2. Trended Signal Animation

# strength of the trend:
trend_strength=0.15

sig_trended = smooth_periodic_signal + trend_strength * time

ani_trended = create_fft_animation(time, sig_trended, title_suffix="(With Linear Trend)")
HTML(ani_trended.to_jshtml())

the reconstruction looks great no - but look at the Fourier analysis... there is spectral leakage across the other frequencies...  

<div style="padding: 15px; border: 1px solid #1565C0; border-left: 6px solid #1565C0; background-color: #E3F2FD; color: #0D47A1; border-radius: 4px;">
<strong> Exercise</strong><br>
Rerun the cell increasing the strength of the trend to e.g. 0.2, 0.3, 0.4 etc... At which trend strength does it become hard to identify the underlying periodicity of the data?
</div>



Do you agree then that we need to preprocess the signal to address this?



## Handling Non-Periodic Signals with Fourier Analysis

When applying the Fourier Transform to a finite, non-periodic timeseries of length $T$, we must bridge the gap between the theory which assumes infinite periodicity and the dataset on hand. For a non-periodic signal of length $T$, we assume $\omega_0 = \frac{2\pi}{T}$, that is, we treat the data as **one single period** of an infinitely repeating wave. This means the lowest frequency the method can resolve is defined strictly by the length of your dataset.

To prevent mathematical artifacts (like spectral leakage and edge spikes), we follow these steps before running the FFT:

| Step | Action | Why? |
| :--- | :--- | :--- |
| **1. De-meaning** | $x_{anomaly} = x - \bar{x}$ | Remove the Offset Prevents $0$ Hz energy from "bleeding" into nearby frequencies (spectral leakage)|
| **2. Detrending** | $x_{flat} = x - (mt + c)$ | Removes the linear slope. Ensures start/end points are at the same vertical level. |
| **3. Tapering** | $x_{win} = x \cdot \text{Window}$ | Uses a **Hann** or **Hamming** window to "fade in" and "out" the edges to zero. |

NOTE: Often detrending routines work by removing the best least-squares regression fit (as introduced earlier in this course) and thus also remove the mean in a combined step. 

The **detrending** is needed to avoid artificial "discrete" jumps where the start and the end of the series do not match up when "cycling" the signal. This creates a signal that is continuous... but is still unlikely to be infinitely differentiable, i.e. the slopes and higher order derivatives are unlikely to match.  This again causes the Gibbs phenomenom and "ringing", so we need to smooth things a little.  This is why **Tapering** (Windowing) is also needed. 

### The Tapering Process: Fading the Edges

**Tapering** (also known as **Windowing**) is the process of multiplying your timeseries by a mathematical "envelope" that forces the start and end of the signal to zero. It is a simple point-by-point multiplication, where the signal $x[n]$ is multiplied it by a "Window Function" $w[n]$ of the same length:

$$x_{tapered}[n] = x_{original}[n] \cdot w[n]$$

* **The Fade-In (Start):** The window starts at (or near) $0$ and gradually rises to $1$, gently introducing the signal.
* **The Center:** The window stays near $1$, leaving the core data untouched.
* **The Fade-Out (End):** The window drops back to $0$, ensuring the signal "lands" softly at the baseline.

#### Common Window Formulas

**The Hann Window (Raised Cosine)**

The Hann window is the standard method for reducing spectral leakage. It starts and ends at exactly zero, creating the smoothest possible transition.

$$w(n) = 0.5 \left( 1 - \cos\left( \frac{2\pi n}{N-1} \right) \right)$$

* **Best for:** General-purpose analysis. It completely eliminates the "cliff" at the seam, ensuring the cleanest frequency spectrum.

**The Hamming Window**

The Hamming window is a variation designed to minimize the "nearest" side-lobes. Unlike the Hann, it does **not** drop all the way to zero (it hits approx. $0.08$ at the boundaries).

$$w(n) = 0.54 - 0.46 \cos\left( \frac{2\pi n}{N-1} \right)$$

* **Best for:** Distinguishing two frequencies that are very close to each other. It provides a slightly sharper "peak" but leaves a tiny bit more noise at the edges.

<div style="padding: 15px; border: 1px solid #1565C0; border-left: 6px solid #1565C0; background-color: #E3F2FD; color: #0D47A1; border-radius: 4px;">
<strong> Discussion</strong><br>
What is one of the issues you can think of with both of these windows?
</div>

### The Tukey Window (The "Tapered Cosine")

If you are concerned that the **Hann** window affects too much of your data (since it scales everything except the exact center), the **Tukey Window** is an alternative. It allows you to define exactly how much of the "burn-in" and "burn-out" you want, leaving the middle of your signal untouched.

The Tukey window is defined by a parameter $\alpha$ (alpha), which represents the fraction of the window that is tapered ($0 \le \alpha \le 1$).

For a signal of length $N$ and indices $n = 0, 1, \dots, N-1$:

$$
w(n) = 
\begin{cases} 
\frac{1}{2} \left[ 1 + \cos\left( \pi \left( \frac{2n}{\alpha(N-1)} - 1 \right) \right) \right] & 0 \le n < \frac{\alpha(N-1)}{2} & \text{(Fade-In)} \\
1 & \frac{\alpha(N-1)}{2} \le n \le (N-1)(1 - \frac{\alpha}{2}) & \text{(Flat Top)} \\
\frac{1}{2} \left[ 1 + \cos\left( \pi \left( \frac{2(N-1-n)}{\alpha(N-1)} - 1 \right) \right) \right] & (N-1)(1 - \frac{\alpha}{2}) < n \le N-1 & \text{(Fade-Out)}
\end{cases}
$$

#### Choosing the Taper Amount ($\alpha$)

The $\alpha$ parameter controls the "aggressiveness" of the taper:
* **$\alpha = 0$**: Becomes a **Rectangular (Boxcar)** window (no tapering at all).
* **$\alpha = 1$**: Becomes a **Hann** window (tapers the entire dataset).
* **$\alpha = 0.1$**: Only the first 5% and last 5% of the data are tapered. The **middle 90% stays multiplied by 1.0**.

Let's compare these windows:

In [ ]:
from scipy.signal.windows import tukey, hann, hamming

# 1. Setup Parameters
x = np.linspace(0, 1, N)
alpha_val = 0.2

# 2. Generate the Windows
# These are the weighting arrays that we multiply our signal by
win_hann = hann(N)
win_hamming = hamming(N)
win_tukey = tukey(N, alpha=alpha_val)
win_boxcar = np.ones(N)  # This is the "No Window" baseline

# 3. Create the Comparison Plot
plt.figure(figsize=(10, 5))

# Plotting the three competitors
plt.plot(x, win_hann, label='Hann Window', color='tab:red', lw=2)
plt.plot(x, win_hamming, label='Hamming Window', color='tab:blue', lw=2)
plt.plot(x, win_tukey, label=f'Tukey Window (alpha={alpha_val})', 
         color='tab:green', lw=3)
plt.plot(x, win_boxcar, label='Boxcar - no window', color='tab:grey', lw=2)

# Highlight the 80% untouched data for the Tukey
plt.fill_between(x, win_tukey, alpha=0.1, color='green')

# Formatting 
plt.title(f"Comparison of Window Functions (N={N})", fontsize=14)
plt.xlabel("Normalized Time (n/N)", fontsize=12)
plt.ylabel("Multiplier (Weight)", fontsize=12)
plt.axhline(1.0, color='black', lw=0.5, ls='--')
plt.axhline(0.0, color='black', lw=0.5, ls='--')

plt.ylim(-0.1, 1.2)
plt.legend(loc='lower center', frameon=True)
plt.grid(alpha=0.3)
plt.tight_layout()

plt.show()

We can look at the spectral leakage by padding the windows themselves with a lot of zeros and then applying the FFT directly to them. 

In [ ]:
from scipy.fft import fft, fftshift, fftfreq

# --- Spectral Leakage Calculation ---

# 1. Zero-Padding
pad_factor = 16 
N_fft = N * pad_factor 

def get_decibel_spectrum(window, n_fft):
    """Calculates the normalized frequency response in dB."""
    # FFT + Zero-Padding in the second option 
    W = fft(window, n=n_fft)
    # Shift center, get magnitude, and normalize peak to 1.0 (0 dB)
    mag = np.abs(fftshift(W))
    mag /= np.max(mag)
    # Convert to dB (log scale)
    return 20 * np.log10(mag + 1e-10)

# Calculate the spectrums for each window
freqs = fftshift(fftfreq(N_fft))
spec_hann    = get_decibel_spectrum(win_hann, N_fft)
spec_hamming = get_decibel_spectrum(win_hamming, N_fft)
spec_tukey   = get_decibel_spectrum(win_tukey, N_fft)
spec_boxcar  = get_decibel_spectrum(win_boxcar, N_fft)

# 3. Create the Spectral Leakage Plot
plt.figure(figsize=(10, 5))

plt.plot(freqs, spec_boxcar,  label='Boxcar (Alpha=0)', color='tab:grey', alpha=0.4, ls='--')
plt.plot(freqs, spec_hann,    label='Hann',           color='tab:red',   lw=2)
plt.plot(freqs, spec_hamming, label='Hamming',        color='tab:blue',  lw=2)
plt.plot(freqs, spec_tukey,   label=f'Tukey (Alpha={alpha_val})', color='tab:green', lw=2.5)

# Formatting for Spectral Analysis
plt.title("Spectral Leakage (Frequency Domain)", fontsize=14)
plt.ylabel("Magnitude (Decibels)", fontsize=12)
plt.xlabel("Normalized Frequency", fontsize=12)
plt.xlim(0, 0.05)   # Zooming in on the "leakage zone" near the peak
plt.ylim(-100, 5)   # Looking deep into the noise floor
plt.grid(True, which='both', linestyle='--', alpha=0.4)
plt.legend()
plt.tight_layout()
plt.show()

### Hann vs. Hamming: The issue of Close Peaks.

Why does anyone still use the **Hamming** window if the **Hann** window has a cleaner overall spectrum? It comes down to **"Near-Side-Lobe Suppression."**

| Window | First Side-Lobe (The 1st Ripple) | Ultimate Noise Floor | Best Use Case |
| :--- | :--- | :--- | :--- |
| **Hamming** | -42 dB (Quieter near the peak) | -40 dB (Constant hum) | Finding a tiny signal hidden right next to a giant one. |
| **Hann** | -31 dB (Louder near the peak) | -100+ dB (Deep silence) | General purpose; cleanest overall "noise floor." |

* Use **Hann** for 99% of modern applications. It is mathematically superior for long-range leakage suppression ($1/f^3$ behaviour)
* Use **Hamming** only used for Speech Processing or Radar where you need to see a faint "echo" immediately adjacent to a main pulse signal.

### The Tukey Window: When to Use It?

The **Tukey Window** is the best choice when you want to balance **Spectral Cleanliness** with **Data Integrity**. In many scientific fields (like Seismology, Structural Engineering, or Climate Science), the actual values (amplitudes) of your signal are just as important as the frequencies. The Hann and Hamming windows misrepresent the amplitudes by squashing the middle of your signal, making your peaks look smaller than they really are.

#### Use a Tukey Window ($\alpha = 0.05$ to $0.2$) when:
1. **The Middle Matters:** You have a long recording and you don't want to alter the measurements in the center of your dataset.
2. **Measuring True Power:** You need the **Total Energy** or **Peak Amplitude** in your FFT to match the real-world physical values.
3. **Transient Analysis:** You are looking for short-lived events (spikes, pops, or bursts) that could happen anywhere in the signal and might be missed with a Hann window if they are close to the start or end of the dataset.

#### Avoid a Tukey Window when:
* **The Noise is Extreme:** If your signal is buried in massive interference, the "Side Lobes" (Ringing) of a Tukey might be too high. In that case, sacrifice your data integrity and use a **Hann** window to sink the noise floor as deep as possible.


In [ ]:
from scipy.signal.windows import tukey, hann, hamming
from scipy.signal import detrend
from scipy.fft import fft, fftshift, fftfreq

def sig_preproc(signal,detrend_opt=1,win=1,tukey_a=0.2):
    """ routine to preprocess the """

    # step 1 and 2: Detrend data and remove mean.
    # for now only linear detrending available
    match detrend_opt:
        case 0: 
            sig_detrended=signal
        case 1: # linear 
            sig_detrended=detrend(signal)
        case _: 
            sig_detrended=signal
            
    # Step 3: Apply the Hann Window (Squash the edges to zero)
    N = len(signal)
    match win:
        case 0: # no window
            win=np.ones(N)
        case 1: # hann window
            win=hann(N)
        case 2: # Hamming window
            win=hamming(N)
        case 3:
            win=tukey(N,alpha=tukey_a)
        case _: # no window
            win=np.ones(N)

    sig_final = sig_detrended * win
    
    return sig_final 

In [ ]:
# 3. Cleaned up dirty trended version of clean Signal!!!

sig_clean = sig_preproc(sig_trended)
ani_clean = create_fft_animation(time, sig_clean, title_suffix="(Clean)")
HTML(ani_clean.to_jshtml())


Here we see that at a slower rate, the red dots trace a wave that has a longer wavelength than the blue line. This is known as "aliasing" where the power at one frequency is artifically projected onto a lower frequency due to undersampling.

Okay, that's enough of artificial series, let's try a real world example. Here we will analysis an hourly temperature series for Trieste that spans about ten years, the data is from ERA5 reanalysis.

In [ ]:
#%pip install cftime
import cftime
from datetime import datetime
import xarray as xr

In [ ]:
def plot_fft_spectrum(frequencies, periods, amplitudes):
    """
    Plots the frequency spectrum.

    Args:
        frequencies (np.ndarray): Frequencies (cycles/day).
        periods (np.ndarray): Periods (days).
        amplitudes (np.ndarray): Amplitudes.
    """

    positive_freq_indices = frequencies > 0

    fig, ax = plt.subplots(figsize=(10, 6))  # Create figure and axes

    ax.plot(periods[positive_freq_indices], amplitudes[positive_freq_indices])
    ax.set_xlabel("Period (days)")
    ax.set_ylabel("Amplitude (log scale)")
    ax.set_title("Frequency Spectrum")
    ax.grid(True)
    ax.set_yscale('log')
    ax.set_xscale('log')
    ax.set_xlim(365 * 10,0.1)

    return fig,ax



We will also introduce a couple of new separate plotting routines in order to keep things clean, one for the spectra and one for the timeseries. 


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

def plot_raw_timeseries(times, temperature_data, title="ERA5 2m Temperature", ylabel="Temperature (K)"):
    """
    Creates a clean, formatted time series plot with dates on the x-axis.

    Args:
        times (list or array): Datetime or cftime objects.
        temperature_data (array-like): The temperature values to plot.
        title (str): The title of the plot.
        ylabel (str): The label for the y-axis.
    """
    # Create the figure and axis
    fig, ax = plt.subplots(figsize=(12, 5))
    
    # Plot the data (using a thin line is usually best for dense hourly data)
    ax.plot(times, temperature_data, color='#1f77b4', linewidth=0.8)
    
    # Format the x-axis to handle dates elegantly
    # You can change the format string (e.g., '%Y-%m' for just year and month)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
    
    # Automatically rotate and align the date labels so they don't overlap
    fig.autofmt_xdate() 
    
    # Add labels, title, and a nice grid for readability
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_xlabel("Date", fontsize=12)
    ax.set_ylabel(ylabel, fontsize=12)
    ax.grid(True, linestyle='--', alpha=0.6)
    
    # Remove extra whitespace around the plot
    plt.tight_layout()
    plt.show()

In [ ]:
from scipy.fft import fft, fftfreq 
from scipy.signal import detrend 

def analyze_time_series_fft(time_series, times, units):
    """
    Analyzes a time series using FFT and returns the frequencies, periods, and amplitudes.

    Args:
        time_series (np.ndarray or xr.DataArray): The time series data.
        time_coords (np.ndarray or xr.DataArray): Corresponding time coordinates.

    Returns:
        tuple: (frequencies, periods, amplitudes)
    """
   

    # Calculate the sampling period in days
    time_diffs = np.diff(times)
    if len(np.unique(time_diffs)) != 1:
        raise ValueError("Time coordinates must be evenly spaced.")

    # Determine time units and convert to days
    if isinstance(times[0], datetime):
        time_diff_days = np.mean([td.total_seconds() / (60 * 60 * 24) for td in time_diffs])
    else:
        try:
            # units = time_coords.units
            print (units, " units")
            if 'second' in units.lower():
                print ("sec detected")
                time_diff_days = np.mean(time_diffs) / (60 * 60 * 24)
            elif 'minute' in units.lower():
                time_diff_days = np.mean(time_diffs) / (60 * 24)
            elif 'hour' in units.lower():
                print ("hours detected")
                time_diff_days = np.mean(time_diffs) / 24
            elif 'day' in units.lower():
                time_diff_days = np.mean(time_diffs)
            else:
                time_diff_days = np.mean(time_diffs) / (60 * 60 * 24)  # assume seconds if units not recognized.
        except AttributeError:
            time_diff_days = np.mean(time_diffs) / (60 * 60 * 24)  # assume seconds if no units attribute.
    
    # Detrend the data (this automatically removes the mean as well)
    # scipy.signal.detrend uses a linear fit by default.
    #detrended_series = detrend(time_series)

    # Apply a Hann window to mitigate spectral leakage
    #window = np.hanning(len(time_series))
    #processed_series = detrended_series * window

    processed_series = sig_preproc(time_series)
    
    # --- END PRE-PROCESSING ---

    frequencies, fft_result, amplitudes, phases = fft_scipy(times, time_series)
    ### frq_scipy,cn_scipy,amps_scipy,phase_scipy=fft_scipy(time,smooth_periodic_signal)

    # Compute the FFT
    #fft_result = np.fft.fft(time_series)

    # Calculate frequencies (in cycles per day)
    #frequencies = np.fft.fftfreq(len(time_series), time_diff_days)

    # Calculate periods (in days) - handle zero frequency carefully
    periods = np.zeros_like(frequencies)

    # we only take the positive frequencies as the 
    # input is a real function, not complex.
    positive_freq_indices = frequencies > 0
    periods[positive_freq_indices] = 1 / frequencies[positive_freq_indices]
    periods[frequencies == 0] = np.inf  # Period is infinite for zero frequency

    # Calculate amplitudes
    #amplitudes = np.abs(fft_result)

    return frequencies, periods, amplitudes


let's try that with some real data...

In [ ]:
# Download example data
# url = "https://downloads.psl.noaa.gov/Datasets/era5/hourly/air.2m.2023.nc"
# filename = "air.2m.2023.nc"
# urllib.request.urlretrieve(url, filename)
import xarray as xr
import cftime
from datetime import datetime

datadir="/Users/tompkins/DATA/era5/hourly/"
filename=datadir+"t2m_t.nc"

# Open the NetCDF file
decode_times=False
ds = xr.open_dataset(filename, decode_times=decode_times)
if not decode_times:
    print ("decode")
    time_units = ds["valid_time"].units
    time_values = ds["valid_time"].values

    # Extract the base date from the time units
    base_date_str = time_units.split('since ')[1]

    # Convert seconds to datetime objects
    times = cftime.num2date(time_values, units = time_units, calendar = 'standard')

    # Replace the time coordinate
    # ds['time'] = datetimes

# Select a specific location and time range
lat = 45.6
lon = 13.7
t2m = ds["t2m"].sel(latitude=lat, longitude=lon,method='nearest')
# time = ds['time']

temperature_values = t2m.values

# Convert cftime objects to standard Python datetimes
stimes = [
    datetime(t.year, t.month, t.day, t.hour, t.minute, t.second) 
    for t in times
]
plot_raw_timeseries(stimes, temperature_values)

So... notice we have complete years here, so the end of the dataseries almost matches the start.  And because of the relatively short series (and the fact that, unintentionally, the data period just so happens to be from ten years where natural climate variability through enhanced ocean heat uptake worked against human-induced change to result in the so-called, and somewhat controversial climate "hiatus") there is relatively little trend, therefore our data treatment of detrending and smoothing is less essential in this series. 

<div class="alert alert-info">

Discussion: what frequencies do we expect to see a "peak" at and why?

</div>


In [ ]:
# Determine conversion factor to Days
# ERA5 'valid_time' is usually "seconds since..." 
# but occasionally "hours since..."
if 'second' in time_units.lower():
    to_days = 1 / (60 * 60 * 24)
elif 'hour' in time_units.lower():
    to_days = 1 / 24
else:
    to_days = 1  # Assume days if not specified
    
# Perform FFT analysis
frequencies, periods, amplitudes = analyze_time_series_fft(t2m, time_values, time_units)
print (periods)

# Determine conversion factor to Days
# ERA5 'valid_time' is usually "seconds since..." 
# but occasionally "hours since..."
if 'second' in time_units.lower():
    to_days = 1 / (60 * 60 * 24)
elif 'hour' in time_units.lower():
    to_days = 1 / 24
else:
    to_days = 1  # Assume days if not specified
# Convert results to DAYS automatically
# (Since frequencies are 1/period, we multiply one and divide the other)
frequencies_days = frequencies / to_days
periods_days = periods * to_days

fig,ax=plot_fft_spectrum(frequencies_days, periods_days, amplitudes)

# expected peaks (in days)
hdaylist=[365,1,0.5,0.333]

for hday in hdaylist:
  ax.axvline(x=hday, color='red', linestyle='--',alpha=0.5)
  ax.text(hday,1,str(hday),color='red')

### Discussion

- What do you notice about the general slope, what period general has more power?  This is an example of a **red** spectrum
- Do you notice any spectral peaks that stand out?  Are they there by chance do you think, or do they have physical significance?  Did you expect these peaks?
- What about the sub-daily peaks?


 #### Exercises
<div class="alert alert-info ">
    
1. Try out the Tukey filter with various window sizes.
2. Add a serious trend to the data to see the impact on the FFT output
3. Try the FFT analysis on your favourite dataset (e.g. Nino3.4 index)
</div> 